# Aristos lens backtest

Runs the real Aristos lenses over the last 10 years on your watched cohorts and writes one CSV per cohort and lens.

**Before you start, put two things on your Google Drive** (from your PC's repo folder):

1. Copy the folder `data\local\cohorts` to `MyDrive/aristos/cohorts` (it holds the frozen member lists, which are not in GitHub).
2. The file `data\local\cohorts\watch.yaml` will come with it; the notebook reads the 13 watched cohort names from there.

Then run the cells top to bottom. The run takes hours; results are saved as they finish, so you can stop and rerun and it carries on where it left off.

In [ ]:
#@title 1. Get the code and install it
BRANCH = "main"  #@param {type:"string"}
# If backtest.py (or whatever you're testing) is not on main yet, change BRANCH to the PR's
# branch name instead.
import os, subprocess, sys
if not os.path.exists("/content/aristos-council"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://github.com/kayvonsalari/aristos-council"], check=True, cwd="/content")
else:
    subprocess.run(["git", "pull"], cwd="/content/aristos-council")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/content/aristos-council"], check=True)
os.chdir("/content/aristos-council")
print("code ready, branch", BRANCH)

In [ ]:
#@title 2. Connect Google Drive (cohorts in, results and cache out)
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import glob, os

DRIVE = "/content/drive/MyDrive/aristos"
OUT = f"{DRIVE}/backtests"                 # the CSVs land here
CACHE = f"{DRIVE}/aristos_cache"           # so a second run is fast and free
os.makedirs(OUT, exist_ok=True); os.makedirs(CACHE, exist_ok=True)

# Find your copy of data/local/cohorts (it holds watch.yaml, which is not in GitHub) wherever it
# sits on the Drive, instead of requiring one fixed path.
candidates = [p for p in glob.glob("/content/drive/MyDrive/**/watch.yaml", recursive=True)
             if "cohorts" in p.replace(os.sep, "/").lower()]
assert candidates, ("no watch.yaml found under MyDrive with 'cohorts' in its path — copy "
                   "data/local/cohorts (it comes with watch.yaml) to your Drive first")
COHORTS_ROOT = os.path.dirname(sorted(candidates)[0])
print("COHORTS_ROOT ->", COHORTS_ROOT)

# the code caches under .aristos_cache in its own folder; point that at Drive
if os.path.islink(".aristos_cache") or os.path.exists(".aristos_cache"):
    import shutil; shutil.rmtree(".aristos_cache", ignore_errors=True)
    if os.path.islink(".aristos_cache"): os.unlink(".aristos_cache")
os.symlink(CACHE, ".aristos_cache")
print("drive ready")

In [ ]:
#@title 3. Your EODHD key (typed, never saved in the notebook)
import os
from getpass import getpass
os.environ["EODHD_API_KEY"] = getpass("EODHD API key: ").strip()
print("key set" if os.environ["EODHD_API_KEY"] else "NO KEY")

In [ ]:
#@title 4. What to test: the watched cohorts and the voting stock lenses
import yaml
watched = yaml.safe_load(open(f"{COHORTS_ROOT}/watch.yaml"))["watch"]

# Explicit, not auto-detected from strategies/*.yaml: an ETF lens (its factors don't mean the
# same thing for a fund), a screen LENS (criteria:-shaped, not a rank strategy) or a CHECK lens
# (marks, does not vote — e.g. forensic_v1) must never be backtested here. Edit this list to test
# a subset or add a lens.
LENSES = ["magic_formula_raw_v1", "magic_formula_momentum_v1", "growth_garp_v2",
         "conservative_plus_v1", "cyclical_income_v1"]

print(len(watched), "cohorts:"); [print("  ", c) for c in watched]
print("\n", len(LENSES), "voting stock lenses:"); [print("  ", l) for l in LENSES]
print(f"\n{len(watched)} cohorts x {len(LENSES)} lenses = {len(watched) * len(LENSES)} runs")

In [ ]:
#@title 5. Run (hours; luck baseline adds ~15s/cohort-lens on a warm cache; safe to stop and rerun, finished pairs are skipped)
YEARS = 10       #@param {type:"integer"}
HOLD_MONTHS = 12 #@param {type:"integer"}
COST_BPS = 50    #@param {type:"number"}
RANDOM_BASKETS = 500  #@param {type:"integer"}
import time, traceback
from datetime import date
import sys; sys.path.insert(0, "/content/aristos-council/src")   # an editable install isn't
                                                                  # visible to an already-running
                                                                  # Colab kernel
from aristos_council.backtest import run_lens_backtest, verdict, to_csv, csv_path, cohort_slug

today = date.today()
end = date(today.year, today.month, 1)                       # last full month
start = date(end.year - YEARS, end.month, 1)
print(f"window {start} -> {end}, hold {HOLD_MONTHS} months, cost {COST_BPS} bps, "
     f"{RANDOM_BASKETS} random baskets/round (BACKTEST-1C)")

results, failures = {}, {}
for cohort in watched:
    for lens in LENSES:
        target = os.path.join(OUT, cohort_slug(cohort), f"{lens}.csv")
        if os.path.exists(target):
            print(f"skip (done): {cohort} x {lens}"); continue
        t0 = time.time()
        print(f"\n=== {cohort} x {lens}")
        try:
            r = run_lens_backtest(cohort, lens, start=start, end=end, hold_months=HOLD_MONTHS,
                                  cost_bps=COST_BPS, cohorts_root=COHORTS_ROOT,
                                  random_baskets=RANDOM_BASKETS,
                                  progress=lambda m: print("   ", m))
            path = to_csv(r, OUT)
            results[(cohort, lens)] = r
            print(f"    -> {verdict(r)} | {path} | {time.time()-t0:.0f}s")
        except Exception as exc:
            failures[(cohort, lens)] = f"{type(exc).__name__}: {exc}"
            print("    FAILED:", failures[(cohort, lens)]); traceback.print_exc(limit=2)
print("\ndone.", len(results), "new results,", len(failures), "failures")
for k, v in failures.items(): print("  ", k, v)

In [ ]:
#@title 6. Summary table: one line per cohort and lens, with the verdict
import glob, pandas as pd
from aristos_council.backtest import (from_csv, verdict, multiple_testing_for_directory,
                                      calibration_report_for_directory)
rows = []
for path in sorted(glob.glob(f"{OUT}/*/*.csv")):
    r = from_csv(path); s = r.summary
    ordered = sorted(r.rounds, key=lambda rnd: rnd.date)
    rows.append({"cohort": r.cohort, "lens": r.lens_id, "verdict": verdict(r, max_luck=r.max_luck),
                 "mean excess / yr": None if s.mean_annual_excess is None else round(100 * s.mean_annual_excess, 1),
                 # BACKTEST-1B: the plain mean beside the same figure with each round's best-returning
                 # BUY name dropped - a wide gap means the lens is proven on one stock, not a basket.
                 "mean excess / yr, drop best": (None if s.mean_annual_excess_drop_best is None
                                                 else round(100 * s.mean_annual_excess_drop_best, 1)),
                 "years positive": f"{s.years_positive} of {s.years_measured}",
                 "hit rate %": None if s.hit_rate is None else round(100 * s.hit_rate),
                 "rounds held": f"{s.n_positions} of {s.n_rounds}",
                 # BACKTEST-1B: the as-of-size-floor-eligible universe size, first round -> last round
                 # - how thin (or not) the cohort was early versus by the end of the window.
                 "n_eligible (first -> last)": (f"{ordered[0].n_eligible} -> {ordered[-1].n_eligible}"
                                                if ordered else "n/a"),
                 "worst round %": None if s.worst_round_excess is None else round(100 * s.worst_round_excess, 1),
                 # BACKTEST-1C/1D: does this beat a random picker, or just the bar? Low luck % is good;
                 # chance pass rate is how often a random picker "proves" itself on THIS cohort's own
                 # noise; drop-best vs random > 0 means the lens leans on its single best pick LESS
                 # than random picking does. All three None when random_baskets=0. random_mode says
                 # whether the random pickers were turnover-matched (default) or drawn independently
                 # (1C, kept for comparison) - see docs/BACKTEST.md, Skill versus luck.
                 "luck % (lower is better)": (None if s.luck_pct_mean is None
                                              else round(100 * s.luck_pct_mean, 1)),
                 "chance pass rate %": (None if s.luck_pct_pass is None
                                        else round(100 * s.luck_pct_pass, 1)),
                 "drop-best vs random": (None if s.drop_best_vs_random is None
                                         else round(100 * s.drop_best_vs_random, 1)),
                 "random_mode": r.random_mode if r.random_baskets else None})
table = pd.DataFrame(rows).sort_values(["cohort", "lens"])
pd.set_option("display.width", 200); display(table)
print(table["verdict"].value_counts())
print(multiple_testing_for_directory(OUT).sentence())
# BACKTEST-1D - is the luck test itself trustworthy? A warning here means "don't trust these luck
# scores at face value yet", not a verdict on any one lens - see docs/BACKTEST.md.
for line in calibration_report_for_directory(OUT).lines():
    print(line)
table.to_csv(f"{OUT}/SUMMARY.csv", index=False)

In [ ]:
#@title 7. Chart: yearly excess per cohort (one panel per cohort, one bar colour per lens)
import matplotlib.pyplot as plt
from aristos_council.backtest import from_csv
by_cohort = {}
for path in sorted(glob.glob(f"{OUT}/*/*.csv")):
    r = from_csv(path); by_cohort.setdefault(r.cohort, []).append(r)
for cohort, rs in by_cohort.items():
    years = sorted({y for r in rs for y in r.year_excess})
    fig, ax = plt.subplots(figsize=(11, 3.2)); width = 0.8 / max(1, len(rs))
    for i, r in enumerate(rs):
        vals = [100 * r.year_excess.get(y, 0) for y in years]
        ax.bar([x + i * width for x in range(len(years))], vals, width, label=f"{r.lens_id} ({verdict(r)})")
    ax.axhline(0, color="black", lw=0.8); ax.axhline(2, color="grey", lw=0.8, ls="--")
    ax.set_xticks([x + 0.4 for x in range(len(years))]); ax.set_xticklabels(years)
    ax.set_ylabel("excess over cohort, % per year"); ax.set_title(cohort); ax.legend(fontsize=7, ncol=3)
    plt.tight_layout(); plt.show()

In [ ]:
#@title 8. Pack the results for the repo
import shutil
zip_path = shutil.make_archive(f"{DRIVE}/backtests_export", "zip", OUT)
print("zipped to", zip_path)
print("On your PC: unzip it into the repo folder as backtests\\ (so backtests\\<cohort>\\<lens>.csv), then commit.")

## Size floor experiment (SIZE-FLOOR-1)

All five lenses carry the same $5bn floor today. This cell measures whether Magic Formula
RAW and Growth would do better (or worse) at a lower one — **without changing the app's own
floor**: `strategies/*.yaml` is never touched, and this is a completely separate grid from
cells 4-8 above (it does not need them run first). See docs/BACKTEST.md, "Size floor
experiment", for why the as-of floor and the liquidity guard both matter here.

Grid: `[magic_formula_raw_v1, growth_garp_v2]` x `[$1bn, $2bn, $5bn]` x 5 named cohorts
(Diversified Mining, Interactive Media & Gaming, Auto Manufacturers, Grid & Electrical
Machinery, Semiconductors) = 30 runs. Writes one `SIZE_FLOOR_SUMMARY.csv`, one row per
(lens, floor, cohort).

In [ ]:
#@title 9. Size floor experiment (SIZE-FLOOR-1): 30 runs, writes SIZE_FLOOR_SUMMARY.csv
# Independent of cells 4-8 above (does not require the main grid to have run first). Same
# Drive, same cache, same EODHD key already set in cell 3. The app's own $5bn floor is NOT
# touched - see docs/BACKTEST.md, "Size floor experiment".
import time, traceback
from datetime import date
import sys; sys.path.insert(0, "/content/aristos-council/src")
import pandas as pd
from aristos_council.backtest import run_lens_backtest, verdict, to_csv, cohort_slug

SIZE_LENSES = ["magic_formula_raw_v1", "growth_garp_v2"]
FLOORS = [1e9, 2e9, 5e9]
# Exact cohort slugs (data/cohort_definitions.yaml), by their display name:
#   materials_diversified_mining, comms_interactive_media_gaming,
#   consumer_auto_manufacturers, industrials_grid_electrical_machinery, tech_semiconductors
SIZE_COHORTS = ["Materials - Diversified Mining", "Comms - Interactive Media & Gaming",
               "Consumer - Auto Manufacturers", "Industrials - Grid & Electrical Machinery",
               "Tech - Semiconductors"]
SIZE_OUT = f"{OUT}/size_floor"
import os; os.makedirs(SIZE_OUT, exist_ok=True)

YEARS = 10
HOLD_MONTHS = 12
COST_BPS = 50
RANDOM_BASKETS = 500
today = date.today()
end = date(today.year, today.month, 1)
start = date(end.year - YEARS, end.month, 1)
print(f"window {start} -> {end}, {len(SIZE_LENSES)} lenses x {len(FLOORS)} floors x "
     f"{len(SIZE_COHORTS)} cohorts = {len(SIZE_LENSES) * len(FLOORS) * len(SIZE_COHORTS)} runs")

size_results, size_failures = {}, {}
for cohort in SIZE_COHORTS:
    for lens in SIZE_LENSES:
        for floor in FLOORS:
            key = (cohort, lens, floor)
            target = os.path.join(SIZE_OUT, cohort_slug(cohort), lens, f"{floor:.0f}.csv")
            if os.path.exists(target):
                print(f"skip (done): {cohort} x {lens} x {floor:,.0f}"); continue
            t0 = time.time()
            print(f"\n=== {cohort} x {lens} x floor {floor:,.0f}")
            try:
                r = run_lens_backtest(cohort, lens, start=start, end=end, hold_months=HOLD_MONTHS,
                                      cost_bps=COST_BPS, cohorts_root=COHORTS_ROOT,
                                      random_baskets=RANDOM_BASKETS, min_cap_usd=floor,
                                      progress=lambda m: print("   ", m))
                path = to_csv(r, target)
                size_results[key] = r
                print(f"    -> {verdict(r)} | {path} | {time.time()-t0:.0f}s")
            except Exception as exc:
                size_failures[key] = f"{type(exc).__name__}: {exc}"
                print("    FAILED:", size_failures[key]); traceback.print_exc(limit=2)
print("\ndone.", len(size_results), "new results,", len(size_failures), "failures")
for k, v in size_failures.items(): print("  ", k, v)

In [ ]:
#@title 10. Size floor summary table: one line per lens, floor and cohort
import glob, pandas as pd
from aristos_council.backtest import from_csv, verdict

rows = []
for path in sorted(glob.glob(f"{SIZE_OUT}/*/*/*.csv")):
    r = from_csv(path); s = r.summary
    ordered = sorted(r.rounds, key=lambda rnd: rnd.date)
    rows.append({
        "cohort": r.cohort, "lens": r.lens_id,
        "floor ($)": None if r.min_cap_usd is None else round(r.min_cap_usd),
        "verdict": verdict(r, max_luck=r.max_luck),
        "mean excess / yr": None if s.mean_annual_excess is None else round(100 * s.mean_annual_excess, 1),
        "years positive": f"{s.years_positive} of {s.years_measured}",
        "hit rate %": None if s.hit_rate is None else round(100 * s.hit_rate),
        "rounds held": f"{s.n_positions} of {s.n_rounds}",
        # as-of-eligible universe size, first round -> last round (BACKTEST-1B, same column
        # the main summary table uses)
        "n_eligible (first -> last)": (f"{ordered[0].n_eligible} -> {ordered[-1].n_eligible}"
                                       if ordered else "n/a"),
        # item 1d - both already computed by the engine; this grid is the first summary row
        # to surface them
        "max drawdown %": None if s.max_drawdown is None else round(100 * s.max_drawdown, 1),
        "n_illiquid (total)": r.total_illiquid,
        "worst round %": None if s.worst_round_excess is None else round(100 * s.worst_round_excess, 1),
        "luck % (lower is better)": None if s.luck_pct_mean is None else round(100 * s.luck_pct_mean, 1),
        "drop-best vs random": (None if s.drop_best_vs_random is None
                                else round(100 * s.drop_best_vs_random, 1)),
    })
table = pd.DataFrame(rows).sort_values(["lens", "cohort", "floor ($)"])
pd.set_option("display.width", 200); display(table)
print(table["verdict"].value_counts())
table.to_csv(f"{SIZE_OUT}/SIZE_FLOOR_SUMMARY.csv", index=False)
print(f"\nwritten to {SIZE_OUT}/SIZE_FLOOR_SUMMARY.csv")
print("On your PC: copy it into the repo as backtests/size_floor/SIZE_FLOOR_SUMMARY.csv and commit"
     " alongside the per-run CSVs under backtests/size_floor/<cohort>/<lens>/<floor>.csv.")